In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [7]:
np.random.seed(77)

# SUMBER 1: Data toko/cabang (JSON) — tabel referensi kecil
toko_list = ["Toko Magelang", "Toko Yogyakarta", "Toko Semarang", "Toko Solo"]
data_toko = [
    {"toko_id": i + 1, "nama_toko": t, "kota": t.split()[1], "manager": np.random.choice(["Rani", "Joko", "Sari", "Bayu"])}
    for i, t in enumerate(toko_list)
]S
with open("data_toko.json", "w") as f:
    for t in data_toko:
        f.write(json.dumps(t) + "\n")

# SUMBER 2: Data produk (CSV) — tabel referensi kecil
data_produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}",
     "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan"]),
     "harga": int(np.random.choice([50000, 100000, 200000]))}
    for i in range(1, 21)
]
pd.DataFrame(data_produk).to_csv("data_produk.csv", index=False)

# SUMBER 3: Data transaksi triwulan (CSV) — data utama, jauh lebih besar
n = 5000
data_transaksi = pd.DataFrame({
    "trx_id": [f"T{i}" for i in range(n)],
    "toko_id": np.random.randint(1, 5, size=n),
    "product_id": np.random.randint(1, 21, size=n),
    "qty": np.random.randint(1, 6, size=n),
})
data_transaksi.to_csv("data_transaksi_triwulan.csv", index=False)

print("Ketiga sumber data berhasil dibuat:")
print(f"- data_toko.json              : {len(data_toko)} baris")
print(f"- data_produk.csv             : {len(data_produk)} baris")
print(f"- data_transaksi_triwulan.csv : {len(data_transaksi)} baris")

Ketiga sumber data berhasil dibuat:
- data_toko.json              : 4 baris
- data_produk.csv             : 20 baris
- data_transaksi_triwulan.csv : 5000 baris


In [9]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan7/raw
!hdfs dfs -put -f data_transaksi_triwulan.csv /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -put -f data_toko.json /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -put -f data_produk.csv /user/mahasiswa/pertemuan7/raw/
!hdfs dfs -ls /user/mahasiswa/pertemuan7/raw

Found 3 items
-rw-r--r--   1 vboxuser supergroup        599 2026-10-11 20:18 /user/mahasiswa/pertemuan7/raw/data_produk.csv
-rw-r--r--   1 vboxuser supergroup        332 2026-10-11 20:18 /user/mahasiswa/pertemuan7/raw/data_toko.json
-rw-r--r--   1 vboxuser supergroup      61655 2026-10-11 20:18 /user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv


In [10]:
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv",
    header=True, inferSchema=True
)
df_toko = spark.read.json("hdfs://localhost:9000/user/mahasiswa/pSertemuan7/raw/data_toko.json")
df_produk = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_produk.csv",
    header=True, inferSchema=True
)

print("Transaksi:", df_transaksi.count(), "| Toko:", df_toko.count(), "| Produk:", df_produk.count())

Transaksi: 5000 | Toko: 4 | Produk: 20


In [11]:
df_gabungan = df_transaksi.join(df_toko, on="toko_id", how="inner") \
                           .join(df_produk, on="product_id", how="inner")

df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+
|product_id|toko_id|trx_id|qty|      kota|manager|      nama_toko|nama_produk|  kategori| harga|total_penjualan|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+---------------+
|         7|      1|    T0|  5|  Magelang|   Bayu|  Toko Magelang|   Produk-7|   Makanan|200000|        1000000|
|         5|      2|    T1|  5|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-5|Elektronik|200000|        1000000|
|         2|      3|    T2|  2|  Semarang|   Rani|  Toko Semarang|   Produk-2|Elektronik|100000|         200000|
|         2|      2|    T3|  2|Yogyakarta|   Bayu|Toko Yogyakarta|   Produk-2|Elektronik|100000|         200000|
|        18|      4|    T4|  2|      Solo|   Rani|      Toko Solo|  Produk-18|Elektronik|100000|         200000|
+----------+-------+------+---+----------+-------+---------------+-----------+----------+------+

In [12]:
# Meringkas total penjualan per kombinasi toko + produk terlebih dahulu
ringkasan_produk_toko = df_gabungan.groupBy("nama_toko", "nama_produk").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Menerapkan window function untuk mengambil top-3 produk pada setiap toko
window_toko = Window.partitionBy("nama_toko").orderBy(col("total_penjualan").desc())
top3_produk = ringkasan_produk_toko.withColumn("peringkat", row_number().over(window_toko)) \
                                    .filter(col("peringkat") <= 3)

top3_produk.orderBy("nama_toko", "peringkat").show(20)

+---------------+-----------+---------------+---------+
|      nama_toko|nama_produk|total_penjualan|peringkat|
+---------------+-----------+---------------+---------+
|  Toko Magelang|   Produk-7|       46000000|        1|
|  Toko Magelang|   Produk-5|       44400000|        2|
|  Toko Magelang|  Produk-14|       38400000|        3|
|  Toko Semarang|   Produk-8|       49200000|        1|
|  Toko Semarang|   Produk-7|       40400000|        2|
|  Toko Semarang|   Produk-5|       39000000|        3|
|      Toko Solo|  Produk-14|       38400000|        1|
|      Toko Solo|   Produk-7|       37200000|        2|
|      Toko Solo|   Produk-5|       33200000|        3|
|Toko Yogyakarta|  Produk-14|       48200000|        1|
|Toko Yogyakarta|   Produk-5|       37600000|        2|
|Toko Yogyakarta|   Produk-4|       37000000|        3|
+---------------+-----------+---------------+---------+



In [13]:
df_gabungan.createOrReplaceTempView("gabungan")

kontribusi_manager = spark.sql('''
    SELECT kota, manager, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kota, manager
    ORDER BY total_penjualan DESC
''')
kontribusi_manager.show()

+----------+-------+---------------+
|      kota|manager|total_penjualan|
+----------+-------+---------------+
|  Semarang|   Rani|      416250000|
|  Magelang|   Bayu|      408550000|
|Yogyakarta|   Bayu|      389150000|
|      Solo|   Rani|      369950000|
+----------+-------+---------------+



In [14]:
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan"
)

print("Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori produk.")
!hdfs dfs -ls /user/mahasiswa/pertemuan7/processed/laporan_triwulan

[Stage 32:>                                                         (0 + 1) / 1]

Laporan berhasil disimpan ke HDFS dalam format Parquet, terpartisi per kategori produk.


Found 4 items
-rw-r--r--   3 vboxuser supergroup          0 2026-10-11 20:18 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/_SUCCESS
drwxr-xr-x   - vboxuser supergroup          0 2026-10-11 20:18 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Elektronik
drwxr-xr-x   - vboxuser supergroup          0 2026-10-11 20:18 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Fashion
drwxr-xr-x   - vboxuser supergroup          0 2026-10-11 20:18 /user/mahasiswa/pertemuan7/processed/laporan_triwulan/kategori=Makanan


In [15]:
df_final = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan")

ringkasan_eksekutif = df_final.groupBy("nama_toko", "kota", "manager").agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif.show()

+---------------+----------+-------+------------------------+----------------+-------------------------+
|      nama_toko|      kota|manager|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------------+----------+-------+------------------------+----------------+-------------------------+
|  Toko Semarang|  Semarang|   Rani|               416250000|            1306|       318721.28637059726|
|  Toko Magelang|  Magelang|   Bayu|               408550000|            1315|       310684.41064638784|
|Toko Yogyakarta|Yogyakarta|   Bayu|               389150000|            1200|        324291.6666666667|
|      Toko Solo|      Solo|   Rani|               369950000|            1179|        313782.8668363019|
+---------------+----------+-------+------------------------+----------------+-------------------------+



In [16]:
spark.stop()
print("SparkSession ditutup.")S

SparkSession ditutup.


---
## Latihan Mandiri

In [30]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_transaksi_triwulan.csv", header=True, inferSchema=True)
df_toko = spark.read.json("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_toko.json")
df_produk = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/pertemuan7/raw/data_produk.csv", header=True, inferSchema=True)
df_gabungan = df_transaksi.join(df_toko, on="toko_id").join(df_produk, on="product_id")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))
print("Siap untuk latihan. Total baris:", df_gabungan.count())

Siap untuk latihan. Total baris: 5000


**Soal 1.** Menggunakan Spark SQL, tampilkan **kategori produk** dengan total `qty` (jumlah unit terjual, bukan nilai rupiah) tertinggi secara keseluruhan.

In [31]:
# Jawaban Soal 1 di sini
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")

hasil_kategori = spark.sql("""
    SELECT p.kategori, SUM(t.qty) AS total_qty
    FROM transaksi t
    JOIN produk p ON t.product_id = p.product_id
    GROUP BY p.kategori
    ORDER BY total_qty DESC
""")

hasil_kategori.show()

+----------+---------+
|  kategori|total_qty|
+----------+---------+
|Elektronik|     6736|
|   Fashion|     5307|
|   Makanan|     3114|
+----------+---------+



**Soal 2.** Menggunakan window function, tentukan **toko dengan penjualan tertinggi** untuk **masing-masing kategori produk** (top-1 saja, gunakan `row_number()`).

In [32]:
# Jawaban Soal 2 di sini
ringkasan_toko = df_gabungan.groupBy("kategori", "nama_toko").agg(
    spark_sum("qty").alias("total_qty")
)

window_kategori = Window.partitionBy("kategori").orderBy(
    col("total_qty").desc()
)

top_toko = ringkasan_toko.withColumn(
    "peringkat", row_number().over(window_kategori)
).filter(col("peringkat") == 1)

top_toko.orderBy("kategori").show()

+----------+-------------+---------+---------+
|  kategori|    nama_toko|total_qty|peringkat|
+----------+-------------+---------+---------+
|Elektronik|Toko Magelang|     1756|        1|
|   Fashion|Toko Semarang|     1500|        1|
|   Makanan|Toko Magelang|      898|        1|
+----------+-------------+---------+---------+



**Soal 3.** Simpan hasil `ringkasan_eksekutif` (dari Sub-bab 7.4 di atas — silakan buat ulang) ke HDFS sebagai Parquet **tanpa partisi**, pada path `/user/mahasiswa/latihan7/ringkasan_eksekutif`.

In [33]:
# Jawaban Soal 3 di sini
!hdfs dfs -mkdir -p /user/mahasiswa/latihan7

ringkasan_eksekutif.write.mode("overwrite").parquet(
    "hdfs://localhost:9000/user/mahasiswa/latihan7/ringkasan_eksekutif"
)

!hdfs dfs -ls /user/mahasiswa/latihan7/ringkasan_eksekutif

Found 2 items
-rw-r--r--   3 vboxuser supergroup          0 2026-10-11 21:32 /user/mahasiswa/latihan7/ringkasan_eksekutif/_SUCCESS
-rw-r--r--   3 vboxuser supergroup       2087 2026-10-11 21:32 /user/mahasiswa/latihan7/ringkasan_eksekutif/part-00000-08cb1e9e-0e0b-41ca-befd-42910308313d-c000.snappy.parquet


**Soal 4 (Refleksi menyeluruh).** Sebagai persiapan UTS, jelaskan dalam 4-5 kalimat: bagaimana **HDFS, Spark, dan format Parquet saling melengkapi** dalam satu ekosistem Big Data? Gunakan istilah-istilah yang telah anda pelajari sejak Pertemuan 3 (mis. distributed storage, in-memory processing, columnar format, partitioning).

Hadoop HDFS digunakan untuk menyimpan data secara terdistribusi sehingga data dapat dikelola dalam sistem penyimpanan yang terdiri dari beberapa bagian. Apache Spark digunakan untuk mengolah dan menganalisis data dengan memanfaatkan pemrosesan dalam memori (*in-memory processing*) agar proses pengolahan dapat berjalan lebih efisien. Format Parquet menyimpan data secara kolumnar sehingga pembacaan kolom tertentu dapat dilakukan dengan lebih efisien. Pembagian data menggunakan partisi dapat membantu mempercepat pembacaan data berdasarkan kolom tertentu, terutama ketika data berukuran besar. Ketiga teknologi tersebut saling melengkapi dalam proses penyimpanan, pengolahan, dan analisis Big Data.

In [ ]:
import numpy as npS
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

### Instruksi Pengerjaan

Buat notebook baru **`Tugas7_[NPM]_[Nama Lengkap].ipynb`**, lalu bangun pipeline analisis lengkap:

---

**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.

**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas7_[NPM]_[Nama Lengkap].ipynb` melalui Asprak, paling lambat **sebelum Pertemuan UTS dimulai**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Extract | Ketiga sumber berhasil dibaca dari HDFS | 10% |
| B. Join & Transformasi | Join benar, kolom `total_penjualan` tepat | 20% |
| C. Window Function | Top-2 menu per cabang benar menggunakan `row_number()` | 20% |
| D. Spark SQL | Kueri SQL murni berjalan benar sesuai instruksi | 20% |
| E. Load | Data tersimpan ke HDFS sebagai Parquet terpartisi & terverifikasi | 15% |
| F. Ringkasan & Rekomendasi | Insight berbasis data, rekomendasi logis dan relevan | 15% |


In [22]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Latihan7") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark aktif:", not spark.sparkContext._jsc.sc().isStopped())

Spark aktif: True


In [27]:
print(ringkasan_eksekutif.count())
ringkasan_eksekutif.show()

Py4JJavaError: An error occurred while calling o137.count.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.wa

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg

spark = SparkSession.builder \
    .appName("Latihan7") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

df_final = spark.read.parquet(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan7/processed/laporan_triwulan"
)

ringkasan_eksekutif = df_final.groupBy(
    "nama_toko", "kota", "manager"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan_triwulan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan_triwulan").desc())

ringkasan_eksekutif.show()

+---------------+----------+-------+------------------------+----------------+-------------------------+
|      nama_toko|      kota|manager|total_penjualan_triwulan|jumlah_transaksi|rata_rata_nilai_transaksi|
+---------------+----------+-------+------------------------+----------------+-------------------------+
|  Toko Semarang|  Semarang|   Rani|               416250000|            1306|       318721.28637059726|
|  Toko Magelang|  Magelang|   Bayu|               408550000|            1315|       310684.41064638784|
|Toko Yogyakarta|Yogyakarta|   Bayu|               389150000|            1200|        324291.6666666667|
|      Toko Solo|      Solo|   Rani|               369950000|            1179|        313782.8668363019|
+---------------+----------+-------+------------------------+----------------+-------------------------+



In [29]:
!hdfs dfs -mkdir -p /user/mahasiswa/latihan7

ringkasan_eksekutif.write.mode("overwrite").parquet(
    "hdfs://localhost:9000/user/mahasiswa/latihan7/ringkasan_eksekutif"
)

!hdfs dfs -ls /user/mahasiswa/latihan7/ringkasan_eksekutif

Found 2 items
-rw-r--r--   3 vboxuser supergroup          0 2026-10-11 21:31 /user/mahasiswa/latihan7/ringkasan_eksekutif/_SUCCESS
-rw-r--r--   3 vboxuser supergroup       2087 2026-10-11 21:31 /user/mahasiswa/latihan7/ringkasan_eksekutif/part-00000-6913d773-9dde-473c-8251-2cb48b3cf39d-c000.snappy.parquet
